# 00 — Data Cleaning & Export
**Spotify Dataset 1921–2020** · ~600k tracks · Source: [Kaggle](https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks)

### Purpose
This notebook is the **single source of truth** for data preparation.  
It runs once and exports a clean, dtype-safe `tracks_clean.parquet` file  
that all downstream notebooks load directly — no repeated cleaning needed.

### Pipeline overview
| Step | Description |
|------|-------------|
| 1 | Load raw `tracks.csv` |
| 2 | Basic cleaning (nulls, duplicates, dtypes) |
| 3 | Feature engineering (date parsing, duration, key/mode labels) |
| 4 | Outlier removal (IQR and Log+IQR) |
| 5 | Noise filtering (non-music tracks) |
| 6 | Derived columns (year, decade) |
| 7 | Export to `data/processed/tracks_clean.parquet` |


## 0 · Imports & Configuration

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from pathlib import Path

from shared.config import PATHS, PALETTE, set_style

set_style()

---
## 1 · Load Raw Data

In [2]:
# [STEP 1] - Load raw tracks CSV
tracks = pd.read_csv(PATHS['raw'])

print(f"Shape: {tracks.shape}")
tracks.head()

Shape: (586672, 20)


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.4450,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.2630,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.1770,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.321,0.0946,7,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],['3BiJGZsyX9sJchTqcSA7Su'],1922,0.402,0.1580,3,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4


In [3]:
# [STEP 2] - Quick overview: dtypes and null counts
tracks.info()

<class 'pandas.DataFrame'>
RangeIndex: 586672 entries, 0 to 586671
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                586672 non-null  str    
 1   name              586601 non-null  str    
 2   popularity        586672 non-null  int64  
 3   duration_ms       586672 non-null  int64  
 4   explicit          586672 non-null  int64  
 5   artists           586672 non-null  str    
 6   id_artists        586672 non-null  str    
 7   release_date      586672 non-null  str    
 8   danceability      586672 non-null  float64
 9   energy            586672 non-null  float64
 10  key               586672 non-null  int64  
 11  loudness          586672 non-null  float64
 12  mode              586672 non-null  int64  
 13  speechiness       586672 non-null  float64
 14  acousticness      586672 non-null  float64
 15  instrumentalness  586672 non-null  float64
 16  liveness          586672 non-nu

In [4]:
# [STEP 3] - Null count per column
tracks.isnull().sum()

id                   0
name                71
popularity           0
duration_ms          0
explicit             0
artists              0
id_artists           0
release_date         0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo                0
time_signature       0
dtype: int64

---
## 2 · Basic Cleaning

### 2.1 Missing values & duplicates

In [5]:
# [STEP 1] - Fill missing track names
tracks['name'] = tracks['name'].fillna('Unknown Track')

# [STEP 2] - Drop exact duplicate track IDs
before = len(tracks)
tracks = tracks.drop_duplicates(subset=['id'])
print(f"Duplicates removed: {before - len(tracks)} rows")

Duplicates removed: 0 rows


### 2.2 Stringified lists → actual lists

The `artists` and `id_artists` columns arrive from CSV as stringified lists  
(e.g. `"['Artist A', 'Artist B']"`). We parse them back into Python lists  
and extract the main (first) artist for convenience.

In [6]:
# [STEP 1] - Strip list symbols and quotes
for col in ['artists', 'id_artists']:
    tracks[col] = (tracks[col]
                   .str.replace('[', '', regex=False)
                   .str.replace(']', '', regex=False)
                   .str.replace("'", '', regex=False))

# [STEP 2] - Split string into actual list
tracks['artists']    = tracks['artists'].str.split(', ')
tracks['id_artists'] = tracks['id_artists'].str.split(', ')

# [STEP 3] - Extract first element as main artist
tracks['main_artist']    = tracks['artists'].str[0]
tracks['id_main_artist'] = tracks['id_artists'].str[0]

# Verify
print(tracks[['artists', 'main_artist']].tail())

                            artists   main_artist
586667                    [阿YueYue]       阿YueYue
586668                 [ROLE MODEL]    ROLE MODEL
586669                    [FINNEAS]       FINNEAS
586670  [Gentle Bones, Clara Benin]  Gentle Bones
586671                  [Afrosound]     Afrosound


---
## 3 · Feature Engineering

### 3.1 Release date → datetime

Spotify's API returns release dates in three inconsistent formats:
- `YYYY-MM-DD` (full date)
- `YYYY-MM` (year + month only)
- `YYYY` (year only)

We standardize all entries to `YYYY-MM-DD` before converting to datetime.

In [7]:
# [STEP 1] - Inspect anomalous formats (length < 10 chars)
anomaly_mask = tracks['release_date'].str.len() < 10
print("Anomalous date formats found:")
print(tracks.loc[anomaly_mask, 'release_date'].unique())

# [STEP 2] - Pad year-only entries (YYYY → YYYY-01-01)
mask_year = tracks['release_date'].str.len() == 4
tracks.loc[mask_year, 'release_date'] += '-01-01'

# [STEP 3] - Pad year-month entries (YYYY-MM → YYYY-MM-01)
mask_month = tracks['release_date'].str.len() == 7
tracks.loc[mask_month, 'release_date'] += '-01'

# [STEP 4] - Convert to datetime
tracks['release_date'] = pd.to_datetime(tracks['release_date'], errors='coerce')

# [STEP 5] - Report remaining NaT
print(f"\nInvalid dates after parsing: {tracks['release_date'].isna().sum()}")

Anomalous date formats found:
<ArrowStringArray>
[   '1922',    '1923',    '1924',    '1925',    '1926',    '1927',    '1928',
    '1929',    '1930',    '1931',
 ...
 '1973-01', '1976-03', '1981-05', '1983-09', '1985-04', '1996-03', '1980-10',
 '1981-10', '1999-10', '1991-05']
Length: 340, dtype: str

Invalid dates after parsing: 0


### 3.2 Duration: milliseconds → minutes

In [8]:
# [STEP 1] - Convert ms to minutes and rename column
tracks['duration_ms'] = tracks['duration_ms'] / (1000 * 60)
tracks = tracks.rename(columns={'duration_ms': 'duration_min'})

print("Duration range (minutes):")
print(tracks['duration_min'].describe().round(2))

Duration range (minutes):
count    586672.00
mean          3.83
std           2.11
min           0.06
25%           2.92
50%           3.58
75%           4.40
max          93.69
Name: duration_min, dtype: float64


### 3.3 Key & Mode: integer → label

Spotify encodes musical key as an integer (Pitch Class notation: 0=C, 1=C♯, … 11=B)  
and mode as a binary (0=Minor, 1=Major).  
We add human-readable label columns via a web scrape of the Pitch Class table on Wikipedia.

In [9]:
# [STEP 1] - Scrape Pitch Class table from Wikipedia
url = 'https://en.wikipedia.org/wiki/Pitch_class'
headers = {'User-Agent': 'SpotifyAnalysisProject/1.0 (educational use)'}
page = requests.get(url, headers=headers)
print(f"Wikipedia request status: {page.status_code}")

soup  = BeautifulSoup(page.content, 'html.parser')
table = soup.find(class_='wikitable')
rows  = table.find_all('tr')

# [STEP 2] - Parse pitch class → key name
pitch_classes = []
for row in rows[1:]:
    th    = row.find('th')
    td    = row.find('td')
    pitch = th.text.strip().split(',')[0].strip()
    tonal = td.text.strip().split(',')[0].strip()
    pitch_classes.append({'pitch': int(pitch), 'key_name': tonal})

df_keys = pd.DataFrame(pitch_classes)

# [STEP 3] - Join key_name into tracks
tracks = tracks.merge(df_keys, how='left', left_on='key', right_on='pitch')
tracks.drop(columns='pitch', inplace=True)

print("\nKey name sample:")
print(tracks[['key', 'key_name']].drop_duplicates().sort_values('key'))

Wikipedia request status: 200

Key name sample:
    key key_name
0     0        C
2     1       C♯
11    2        D
4     3       D♯
6     4        E
5     5        F
9     6       F♯
3     7        G
16    8       G♯
29    9        A
25   10       A♯
10   11        B


In [10]:
# [STEP 4] - Add mode label (0 → Minor, 1 → Major)
tracks['mode_name'] = tracks['mode'].map({0: 'Minor', 1: 'Major'})

print("Mode distribution:")
print(tracks['mode_name'].value_counts())

Mode distribution:
mode_name
Major    386498
Minor    200174
Name: count, dtype: int64


### 3.4 Derived temporal columns

In [11]:
# [STEP 1] - Extract year and decade
tracks['year']   = tracks['release_date'].dt.year
tracks['decade'] = (tracks['year'] // 10) * 10

# [STEP 2] - Remove the single anomalous record from year 1900
before = len(tracks)
tracks = tracks[tracks['year'] > 1900]
print(f"Removed {before - len(tracks)} pre-1901 records")
print(f"Year range: {tracks['year'].min()} – {tracks['year'].max()}")

Removed 1 pre-1901 records
Year range: 1922 – 2021


---
## 4 · Outlier Removal

### Feature classification

Features are grouped by their statistical properties, which determines  
the appropriate cleaning strategy:

| Strategy | Features | Rationale |
|----------|----------|-----------|
| **Untouched** | `acousticness`, `danceability`, `energy`, `liveness`, `valence`, `speechiness`, `instrumentalness` | Bounded 0–1 by Spotify's design; extreme values are real |
| **IQR** | `tempo` | Roughly symmetric; standard fence sufficient |
| **Log + IQR** | `loudness`, `duration_min` | Skewed/unbounded; log transform normalizes before fencing |

In [12]:
# -- Feature groups for cleaning --
iqr_features     = ['tempo']
log_iqr_features = ['loudness', 'duration_min']

# Loudness is negative — shift to positive before log transform
LOUDNESS_SHIFT = abs(tracks['loudness'].min())

print(f"Loudness shift value: {LOUDNESS_SHIFT:.2f} dB")

Loudness shift value: 60.00 dB


### Cleaning functions

In [13]:
def apply_iqr(df, features, fence=1.5):
    """
    Removes rows where values fall outside IQR-based bounds.
    Best for roughly symmetric distributions.

    Parameters
    ----------
    df       : input DataFrame
    features : list of column names to clean
    fence    : IQR multiplier (1.5 = standard, 3.0 = conservative)
    """
    mask = pd.Series([True] * len(df), index=df.index)

    for col in features:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lb = Q1 - fence * IQR   # lower bound
        ub = Q3 + fence * IQR   # upper bound

        col_mask = (df[col] >= lb) & (df[col] <= ub)
        mask     = mask & col_mask

        print(f"  [IQR] {col}: bounds ({lb:.2f}, {ub:.2f})"
              f" — removed {(~col_mask).sum()} rows")

    return df[mask]

# -----------------------------------------------------------

def apply_log_iqr(df, features, fence=1.5):
    """
    Applies log1p transform, computes IQR bounds on log scale,
    then back-converts bounds to the original scale before filtering.
    Best for right-skewed, positive distributions.

    Parameters
    ----------
    df       : input DataFrame
    features : list of column names to clean
    fence    : IQR multiplier
    """
    mask = pd.Series([True] * len(df), index=df.index)

    for col in features:
        log_col = np.log1p(df[col])

        Q1  = log_col.quantile(0.25)
        Q3  = log_col.quantile(0.75)
        IQR = Q3 - Q1

        lb = np.expm1(Q1 - fence * IQR)   # back to original scale
        ub = np.expm1(Q3 + fence * IQR)

        col_mask = (df[col] >= lb) & (df[col] <= ub)
        mask     = mask & col_mask

        print(f"  [Log+IQR] {col}: bounds ({lb:.2f}, {ub:.2f})"
              f" — removed {(~col_mask).sum()} rows")

    return df[mask]

# -----------------------------------------------------------

def apply_noise_filter(df, keywords, columns=['name']):
    """
    Removes rows where any column contains noise keywords.
    Filters non-music tracks (sleep sounds, medleys, live recordings)
    that would distort audio feature analysis.

    Parameters
    ----------
    df       : input DataFrame
    keywords : list of strings to filter out (case-insensitive)
    columns  : list of column names to search
    """
    pattern = '|'.join(keywords)
    mask    = pd.Series([True] * len(df), index=df.index)

    for col in columns:
        col_mask = ~df[col].str.lower().str.contains(pattern, na=False)
        mask     = mask & col_mask
        print(f"  [Noise] '{col}': removed {(~col_mask).sum()} rows")

    return df[mask]

### Run cleaning pipeline

In [14]:
noise_keywords = [
    'white noise', 'brown noise', 'baby sleep', 'ocean waves',
    'rain sounds', 'meditation', 'shhh', 'ao vivo',
    'previa', 'medley', 'mixtape', 'concert'
]

print('=' * 55)
print('  CLEANING PIPELINE')
print('=' * 55)

tracks_clean = tracks.copy()

# Shift loudness to positive for log transform
tracks_clean['loudness'] = tracks_clean['loudness'] + LOUDNESS_SHIFT

print(f"\n→ Starting rows: {len(tracks_clean):,}")

# [1/3] IQR on tempo
print("\n[1/3] IQR cleaning:")
tracks_clean = apply_iqr(tracks_clean, iqr_features, fence=1.5)

# [2/3] Log+IQR on loudness and duration_min
print("\n[2/3] Log + IQR cleaning:")
tracks_clean = apply_log_iqr(tracks_clean, log_iqr_features, fence=3)

# Restore original loudness scale
tracks_clean['loudness'] = tracks_clean['loudness'] - LOUDNESS_SHIFT

# [3/3] Noise keyword filter
print("\n[3/3] Noise keyword filter:")
tracks_clean = apply_noise_filter(tracks_clean, noise_keywords, columns=['name'])

print("\n" + "=" * 55)
print(f"  DONE — Kept: {len(tracks_clean):,} / {len(tracks):,}"
      f"  ({len(tracks_clean) / len(tracks) * 100:.1f}%)")
print("=" * 55)

  CLEANING PIPELINE

→ Starting rows: 586,671

[1/3] IQR cleaning:
  [IQR] tempo: bounds (34.52, 197.40) — removed 5709 rows

[2/3] Log + IQR cleaning:
  [Log+IQR] loudness: bounds (32.05, 78.34) — removed 3758 rows
  [Log+IQR] duration_min: bounds (0.50, 13.12) — removed 3057 rows

[3/3] Noise keyword filter:
  [Noise] 'name': removed 4607 rows

  DONE — Kept: 569,617 / 586,671  (97.1%)


---
## 5 · Distribution Transforms

`speechiness` and `instrumentalness` are heavily right-skewed.  
A square-root transform reduces skew without distorting the 0–1 range,  
and is used in the clustering pipeline.

In [15]:
# [STEP 1] - Remove valence encoding artifacts (valence == 1.0 are data errors)
before = len(tracks_clean)
tracks_clean = tracks_clean[tracks_clean['valence'] < 1.0].copy()
print(f"Valence artifacts removed: {before - len(tracks_clean)} rows")

# [STEP 2] - Square root transform for right-skewed features
tracks_clean['speechiness_sqrt']       = np.sqrt(tracks_clean['speechiness'])
tracks_clean['instrumentalness_sqrt']  = np.sqrt(tracks_clean['instrumentalness'])

print("\nTransformed columns added: speechiness_sqrt, instrumentalness_sqrt")

Valence artifacts removed: 13 rows

Transformed columns added: speechiness_sqrt, instrumentalness_sqrt


---
## 6 · Export to Parquet

Parquet preserves all dtypes across sessions (including lists, datetimes,  
and categoricals), unlike CSV which collapses everything to strings.  
All downstream notebooks load this file with a single `pd.read_parquet()` call.

In [17]:
# [STEP 1] - Ensure output directory exists
PATHS['processed'].mkdir(parents=True, exist_ok=True)

# [STEP 2] - Export
tracks_clean.to_parquet(PATHS['clean'], index=False)

print(f"✅ Exported to: {PATHS['clean']}")
print(f"   Shape      : {tracks_clean.shape}")
print(f"   File size  : {PATHS['clean'].stat().st_size / 1e6:.1f} MB")

✅ Exported to: C:\Users\Manuel\OneDrive - Axxam S.p.A\Documenti\06_data_analytics\03_boolean_final_project_spotify\spoti-century\data\processed\tracks_clean.parquet
   Shape      : (569604, 28)
   File size  : 54.1 MB


In [18]:
# [STEP 3] - Sanity check: reload and verify dtypes survived
check = pd.read_parquet(PATHS['clean'])

print("Dtype verification after parquet round-trip:")
print(check.dtypes)
print(f"\nLists preserved — artists sample: {check['artists'].iloc[0]}")

Dtype verification after parquet round-trip:
id                                  str
name                                str
popularity                        int64
duration_min                    float64
explicit                          int64
artists                          object
id_artists                       object
release_date             datetime64[us]
danceability                    float64
energy                          float64
key                               int64
loudness                        float64
mode                              int64
speechiness                     float64
acousticness                    float64
instrumentalness                float64
liveness                        float64
valence                         float64
tempo                           float64
time_signature                    int64
main_artist                         str
id_main_artist                      str
key_name                            str
mode_name                          

---
## Summary

| Metric | Value |
|--------|-------|
| Raw rows | ~600,000 |
| Clean rows | see pipeline output above |
| Columns added | `main_artist`, `id_main_artist`, `key_name`, `mode_name`, `year`, `decade`, `speechiness_sqrt`, `instrumentalness_sqrt` |
| Columns modified | `duration_ms` → `duration_min`, `release_date` → datetime |
| Output | `data/processed/tracks_clean.parquet` |

**→ Next notebook:** `01_eda.ipynb` — univariate distributions and dataset overview